# LangGraph 自动生成段子

本 notebook 用 LangGraph 构建一条 **段子创作流水线**：先让本地 SearXNG 搜索引擎搜索全网搞笑段子，保存到 `段子.txt`；再由商汤 SenseNova 模型**学习分析**这些段子的幽默特点，保存到 `段子分析.txt`；最后根据用户要求创作新段子。

## 工作流程

1. **搜索节点**：本地 SearXNG（localhost:6688）搜索全网搞笑段子
2. **保存节点**：把段子素材写入 `段子.txt`
3. **学习节点**：LLM 分析段子的幽默手法、结构、语言风格 → 写入 `段子分析.txt`
4. **生成节点**：根据学习到的特点 + 用户要求，创作新段子

## 图结构

```
START → search（SearXNG 搜索搞笑段子）
      → save_jokes（保存段子到 段子.txt）
      → learn（LLM 分析特点 → 保存到 段子分析.txt）
      → generate（根据用户要求生成段子）
      → END
```

这是一条**线性流水线**，四个节点依次执行，每一步的产物都写入状态（State）传递给下一个节点。

## 1. 准备环境

In [1]:
# 导入依赖
import os
import requests
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

# 商汤平台付费模型（OpenAI 兼容接口 token.sensenova.cn，同一 API Key 计费）
# 注：经测试推理模型 flash-lite 在创作任务上会推理失控（烧光 65535 token
# 后空返回），故选用平台上的 glm-5.2，实测正常收敛
llm = ChatOpenAI(
    model='glm-5.2',
    base_url='https://token.sensenova.cn/v1',
    api_key=os.getenv('SENSENOVA_API_KEY', ''),
    temperature=0.3,
)


def llm_text(resp):
    # 取模型回复文本；推理模型偶发 content 为空时退回 reasoning_content
    return resp.content or (resp.additional_kwargs.get('reasoning_content') or '')


print('LLM 初始化完成')

# 输出文件路径（与本 notebook 同目录）
BASE_DIR = r's:\PythonAILearningCourse\06-langgraph-multi-agent'
JOKES_FILE = os.path.join(BASE_DIR, '段子.txt')
ANALYSIS_FILE = os.path.join(BASE_DIR, '段子分析.txt')
print('段子文件:', JOKES_FILE)
print('分析文件:', ANALYSIS_FILE)

LLM 初始化完成
段子文件: s:\PythonAILearningCourse\06-langgraph-multi-agent\段子.txt
分析文件: s:\PythonAILearningCourse\06-langgraph-multi-agent\段子分析.txt


## 2. 搜索工具（本地 SearXNG）

In [1]:
# 本地搜索引擎（SearXNG，端口 6688）
SEARCH_URL = 'http://localhost:6688/search'
# 国内网络下国外引擎（google/duckduckgo 等）不可达，且默认 general 分类不含
# bilibili/sogou，这里显式指定实测可达的引擎
SEARCH_ENGINES = 'bilibili,sogou'

def search_web(query, top_n=8):
    # 通过本地 SearXNG 搜索引擎查询，返回格式化的段子素材文本
    try:
        resp = requests.get(SEARCH_URL, params={'q': query, 'format': 'json', 'engines': SEARCH_ENGINES}, timeout=20)
        data = resp.json()
        results = data.get('results', [])[:top_n]
        if not results:
            return f'搜索{query}无结果'
        lines = []
        for i, r in enumerate(results, 1):
            title = r.get('title', '')
            content = r.get('content', '')[:300]
            lines.append(f'【段子{i}】{title}\n{content}')
        return '\n\n'.join(lines)
    except Exception as e:
        return f'搜索失败（请确认 localhost:6688 已启动）: {e}'

# 测试搜索引擎
print(search_web('搞笑段子', top_n=3)[:300])

【段子1】圆哥 你 你 你睡觉怎么不穿裤子？#大圆哥搞笑段子
-

【段子2】你上班有遇到过什么奇葩事吗？


【段子3】搞笑段子_网购买家秀



## 3. 定义状态

In [1]:
class JokeState(TypedDict):
    user_request: str    # 用户对段子的要求
    jokes_text: str      # 搜索到的段子素材文本
    analysis: str        # LLM 学习分析的段子特点
    generated_joke: str  # 根据用户要求生成的段子

## 4. 定义节点

In [1]:
def search_node(state: JokeState) -> dict:
    # 搜索节点：用本地 SearXNG 搜索全网搞笑段子
    print('  [search] 正在搜索搞笑段子...')
    jokes_text = search_web('搞笑段子', top_n=8)
    n = jokes_text.count('【段子')
    print(f'  [search] 获取到 {n} 条段子素材')
    return {'jokes_text': jokes_text}

In [1]:
def save_jokes_node(state: JokeState) -> dict:
    # 保存节点：把搜索到的段子写入 段子.txt
    jokes_text = state['jokes_text']
    with open(JOKES_FILE, 'w', encoding='utf-8') as f:
        f.write('=== 全网搞笑段子（SearXNG 搜索结果） ===\n\n')
        f.write(jokes_text)
    print(f'  [save] 段子已保存到 {JOKES_FILE}')
    return {}

In [1]:
def learn_node(state: JokeState) -> dict:
    # 学习节点：LLM 分析段子特点，保存分析结果到 段子分析.txt
    jokes_text = state['jokes_text']
    print('  [learn] 正在分析段子特点...')
    prompt = f'''以下是从网上搜索到的搞笑段子：

{jokes_text[:3000]}

请分析这些段子的特点，从以下几个方面总结：
1. 幽默手法（如反转、夸张、谐音、自嘲、比喻等）
2. 结构特点（铺垫-反转、长短句节奏等）
3. 语言风格（口语化程度、网络用语等）
4. 常见主题（职场、生活、恋爱、家庭等）

请用中文输出分析结果：'''
    resp = llm.invoke(prompt)
    analysis = llm_text(resp)
    with open(ANALYSIS_FILE, 'w', encoding='utf-8') as f:
        f.write('=== 搞笑段子特点分析（模型学习结果） ===\n\n')
        f.write(analysis)
    print(f'  [learn] 分析完成，已保存到 {ANALYSIS_FILE}')
    return {'analysis': analysis}

In [1]:
def generate_node(state: JokeState) -> dict:
    # 生成节点：根据学习到的特点 + 用户要求创作段子
    analysis = state['analysis']
    user_request = state['user_request']
    print(f'  [generate] 根据要求创作段子: {user_request}')
    prompt = f'''你已经学习了以下搞笑段子的特点分析：

{analysis[:2000]}

请根据用户要求创作 2 个新段子。
用户要求：{user_request}

要求：
1. 参考以上分析的幽默手法和结构特点
2. 内容要原创、简短、有趣
3. 每个段子前用【段子一】【段子二】标注

请直接输出段子：'''
    resp = llm.invoke(prompt)
    print('  [generate] 段子生成完成')
    return {'generated_joke': llm_text(resp)}

## 5. 构建图

In [1]:
# 构建线性流水线：搜索 → 保存 → 学习 → 生成
graph = StateGraph(JokeState)

graph.add_node('search', search_node)
graph.add_node('save_jokes', save_jokes_node)
graph.add_node('learn', learn_node)
graph.add_node('generate', generate_node)

graph.add_edge(START, 'search')
graph.add_edge('search', 'save_jokes')
graph.add_edge('save_jokes', 'learn')
graph.add_edge('learn', 'generate')
graph.add_edge('generate', END)

app = graph.compile()
print('图构建完成！')

图构建完成！


## 6. 运行

In [1]:
# 运行：用户要求（固定样例，避免交互输入的 EOFError）
USER_REQUEST = '以程序员加班为主题，创作2个贴近生活的好笑段子'

result = app.invoke({
    'user_request': USER_REQUEST,
    'jokes_text': '',
    'analysis': '',
    'generated_joke': '',
})

print()
print('=' * 60)
print('生成的段子:')
print('=' * 60)
print(result['generated_joke'])

  [search] 正在搜索搞笑段子...
  [search] 获取到 8 条段子素材
  [save] 段子已保存到 s:\PythonAILearningCourse\06-langgraph-multi-agent\段子.txt
  [learn] 正在分析段子特点...
  [learn] 分析完成，已保存到 s:\PythonAILearningCourse\06-langgraph-multi-agent\段子分析.txt
  [generate] 根据要求创作段子: 以程序员加班为主题，创作2个贴近生活的好笑段子
  [generate] 段子生成完成

生成的段子:
【段子一】
千万别让程序员连轴转加班48小时！昨晚我室友改Bug改到凌晨四点，外卖小哥敲门送夜宵，他迷迷糊糊拉开门，对着小哥的脑袋猛戳了两下，还嘀咕：“怎么点不掉，这弹窗有防抖功能？”小哥吓得后退三步：“兄弟，我这是真人NPC，不是广告弹窗！”逻辑闭环，精神稳定。#职场日常 #离谱操作 #程序员日常 #硬核打工人

【段子二】
你上班有遇到过什么奇葩事吗？昨晚我们组后端老李加班到凌晨三点，他老婆打电话查岗问他在干嘛，老李盯着满屏的红色报错，脱口而出：“我在找Bug啊！”他老婆冷笑一声：“怎么，你是在排查你人生中的Bug吗？都加出幻觉了！”老李叹了口气：“何止幻觉，我现在看键盘上的Ctrl和C都觉得它俩在搞对象。”代码跑通，发际线空。#打工人 #地狱笑话 #程序员加班 #真实日常


## 总结

### 流水线设计
- **搜索 → 保存 → 学习 → 生成** 四个节点依次执行，状态（State）在节点间传递
- 搜索和保存节点是普通 Python 函数；学习和生成节点调用商汤 SenseNova 模型
- 两个落盘文件都在本 notebook 同目录：`段子.txt`（素材）、`段子分析.txt`（学习结果）

### 为什么这样设计？
- **先搜集素材再创作**：模型基于真实段子总结的幽默手法，比凭空编造更接地气
- **学习结果落盘**：`段子分析.txt` 可以复用——下次创作时直接读取，不必重新搜索分析
- **用户要求与素材解耦**：`user_request` 只在生成节点生效，换主题只需改这一处